# INSURANCE COMPANY DATA ANALYSIS

## PROBLEM STATEMENT:

One of our clients is an insurance provider that provides health insurance towards its clients' clients. They now need our assistance in developing a model used to determine whether the insured (customers) from the previous year will be interested in the company's vehicle insurance.

An insurance company is a contract whereby a business agrees to guarantee compensation in the event of a specific loss, damage, disease, or death in return for a payment of a predetermined premium. The amount of money which the customer must consistently pay to an insurance provider in exchange for this assurance is known as a premium.

For instance, we might pay a yearly payment of Rs. 500 for a Rs. 20000/- health insurance policy so that, should the unthinkable happen and we become ill and require hospitalization that year, then insurance provider firm will cover the cost of hospitalization, etc. up to Rs. 200000. The idea of possibilities enters the picture when we wonder how the corporation can afford such large hospitalization bills when it only charges a premium of Rs. 5000.

For instance, 100 consumers, like us, may pay a premium of Rs. 5000 a year, but only a small number of them (let's say 2-3) would end up in the hospital that year. In this sense, each person shares in the danger of the others.
Similar to medical insurance, car insurance requires customers to pay an annual premium to the insurance provider firm in order for them to receive compensation (referred to as the "sum assured") in the event that their vehicle causes an unfortunate accident.

The organization can then organize its communication approach to reach out to those clients and maximize its business model & income by using a model to forecast whether a client would be interested in Vehicle Insurance.


# IMPORTING LIBRARIES:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# LOADING DATA 

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [ ]:
df_train.columns

# DATA DICTIONARY

# EXPLORATORY DATA ANALYSIS

In [ ]:
#shape of train data
df_train.shape

In [ ]:
#shape of test data
df_test.shape

In [ ]:
df_train.columns

In [ ]:
df_test.columns

In [ ]:
df_train.head()

In [ ]:
# Check the column-wise info of the dataframe
df_train.info()

In [ ]:
# Get a summary of the dataframe using 'describe()'

df_train.describe()

In [ ]:
df_test.describe()

## DATA CLEANING

In [ ]:
# Get the column-wise Null count using 'is.null()' alongwith the 'sum()' function
df_train.isnull().sum()

In [ ]:
df_test.isnull().sum()

### There are no null values in our data.

### Attributes classification:

# DATA VISUALIZATION

In [ ]:
#finding the number of responses
sns.countplot(x = df_train['Response'])

In [ ]:
#based on driving license who took insurance
sns.countplot(x = df_train['Driving_License'],hue = df_train['Response'])

In [ ]:
#distibution of age group
sns.histplot(x = df_train['Age'])

In [ ]:
#categorizing age groups
df_train['age_group'] = pd.cut(df_train['Age'],bins = [19,40,60,85],labels = ['Adult','Middle_aged','Senior_citizen'])

In [ ]:
df_train.head()

In [ ]:
#dividing age_group based on response

z = df_train.groupby(['age_group','Response']).Response.count()

df1 = pd.DataFrame(z)
df1.rename(columns = {'Response':'Percent'},inplace = True)
df1.reset_index(inplace = True)
print(df1)

x = df1.groupby(['age_group','Response']).agg({'Percent':'sum'})
x_pcts = x.groupby(level = 0).apply(lambda x : 100 * x / float(x.sum()))
x_pcts.reset_index(inplace = True)
x_pcts
sns.barplot(x = x_pcts['age_group'],y = x_pcts['Percent'],hue = x_pcts['Response'])

### By this we conclude that middle_aged people having more vehicle_insurance acceptance

In [ ]:
sns.catplot(data = df_train,x = 'Vehicle_Age',col = 'Vehicle_Damage',hue = 'Response',kind = 'count')

### Most of the damaged vehicles are not opting for insurance

In [ ]:
#creating boxplots of numerical features
numerical_features = ['Age','Annual_Premium','Vintage']
plt.figure(figsize=(16,16))
for i, col in enumerate(numerical_features):
    plt.subplot(4,4,i+1)
    sns.boxplot(x = df_train[col])
    plt.tight_layout()

In [ ]:
#annual premium have more outliers so who have more than 400000 are set to 400000
df_train.loc[df_train.Annual_Premium> 400000,'Annual_Premium']=400000

In [ ]:
sns.countplot(x = df_train['Vehicle_Damage'],hue = df_train['Vehicle_Age'])

In [ ]:
sns.countplot(x = df_train['Vehicle_Age'],hue = df_train['Response'])

### Since the vehicles are provided with a insurance for a year, so it was observed vehicle age between 1-2 years are less opting for vehicle insurance

In [ ]:
z = df_train.groupby(['Vehicle_Damage','Gender']).Gender.count()

df1 = pd.DataFrame(z)
df1.rename(columns = {'Gender':'Percent'},inplace = True)
df1.reset_index(inplace = True)
print(df1)

x = df1.groupby(['Vehicle_Damage','Gender']).agg({'Percent':'sum'})
x_pcts = x.groupby(level = 0).apply(lambda x : 100 * x / float(x.sum()))
x_pcts.reset_index(inplace = True)
x_pcts
sns.barplot(x = x_pcts['Vehicle_Damage'],y = x_pcts['Percent'],hue = x_pcts['Gender'])



### male has more vehicle damage

In [ ]:
z = df_train.groupby(['Gender','Response']).Response.count()

df1 = pd.DataFrame(z)
df1.rename(columns = {'Response':'Percent'},inplace = True)
df1.reset_index(inplace = True)
print(df1)

x = df1.groupby(['Gender','Response']).agg({'Percent':'sum'})
x_pcts = x.groupby(level = 0).apply(lambda x : 100 * x / float(x.sum()))
x_pcts.reset_index(inplace = True)
x_pcts
sns.barplot(x = x_pcts['Gender'],y = x_pcts['Percent'],hue = x_pcts['Response'])


### vehicle damage is more in male and also they are opting more vehicle insurances

In [ ]:
sns.countplot(x = df_train['Vehicle_Damage'])

In [ ]:
sns.boxplot(y = df_train['Annual_Premium'],x = df_train['Vehicle_Damage'])

# MODELLING

### label encoding

In [ ]:
#converting categorical data to numeric data

df_train.loc[df_train['Gender'] == 'Male', 'Gender'] = 1
df_train.loc[df_train['Gender'] == 'Female', 'Gender'] = 0
df_test.loc[df_test['Gender'] == 'Male', 'Gender'] = 1
df_test.loc[df_test['Gender'] == 'Female', 'Gender'] = 0

df_train.loc[df_train['Vehicle_Age'] == '> 2 Years', 'Vehicle_Age'] = 2
df_train.loc[df_train['Vehicle_Age'] == '1-2 Year', 'Vehicle_Age'] = 1
df_train.loc[df_train['Vehicle_Age'] == '< 1 Year', 'Vehicle_Age'] = 0
df_test.loc[df_test['Vehicle_Age'] == '> 2 Years', 'Vehicle_Age'] = 2
df_test.loc[df_test['Vehicle_Age'] == '1-2 Year', 'Vehicle_Age'] = 1
df_test.loc[df_test['Vehicle_Age'] == '< 1 Year', 'Vehicle_Age'] = 0

df_train.loc[df_train['Vehicle_Damage'] == 'Yes', 'Vehicle_Damage'] = 1
df_train.loc[df_train['Vehicle_Damage'] == 'No', 'Vehicle_Damage'] = 0
df_test.loc[df_test['Vehicle_Damage'] == 'Yes', 'Vehicle_Damage'] = 1
df_test.loc[df_test['Vehicle_Damage'] == 'No', 'Vehicle_Damage'] = 0

del df_train['age_group']



In [ ]:
df_train

In [ ]:
for col in df_train.columns:
    df_train[col] = df_train[col].astype(np.int32)

df_train

### correlation coefficients with respect to target variable

In [ ]:
for col in df_train.columns:
    if(col == 'Response'):
        continue
    print(col,df_train[col].corr(df_train['Response']),sep = '  ')

In [ ]:
y = df_train['Response']
X = df_train.drop(['Response'],axis = 1)



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=666)
print('Positive cases % in validation set: ', round(100 * len(y_test[y_test == 1]) / len(y_test), 3), '%')
print('Positive cases % in train set: ', round(100 * len(y_train[y_train == 1]) / len(y_train), 3), '%')

In [ ]:
#Visualization
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Feature Engineering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Model Selection and Metrics
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, reset_parameter
from sklearn.metrics import f1_score, recall_score, accuracy_score, roc_auc_score, precision_score, auc, roc_curve

# Hyperparamter Tuning
from scipy.stats import randint as sp_randint
from scipy.stats import uniform as sp_uniform
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

In [ ]:
model = {
    "Decision Tree": DecisionTreeClassifier(), 
    "SGD" : SGDClassifier(), 
    "Random Forest" : RandomForestClassifier(), 
    "Gradient Boosting" : GradientBoostingClassifier(),
    "XGBoost" : XGBClassifier(),
    "CatBoost" : CatBoostClassifier(),
    "LGBM" : LGBMClassifier()
        }

In [ ]:
scores = []
prob_score = {}
for mod in model:
    classifier = model[mod]
    classifier.fit(X_train, y_train)
    pred = classifier.predict(X_test)
    try:
        score = classifier.predict_proba(X_test)[:,1]
        roc = roc_auc_score(y_test, score, average='weighted')
        prob_score[mod] = score
    except:
        roc = 0
    scores.append([
        mod,
        accuracy_score(y_test, pred),
        f1_score(y_test, pred, average='weighted'),
        precision_score(y_test, pred, average='weighted'),
        recall_score(y_test, pred, average='weighted'),
        roc
    ])

In [ ]:
scores

In [ ]:
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: yellow' if v else '' for v in is_max]

scores_df  = pd.DataFrame(scores)
index_model = {count: s for count, s in enumerate(scores_df[0])}
col = {count+1: s for count, s in enumerate(['Accuracy','F1 Score','Precision','Recall','ROC AUC'])}
scores_df = scores_df.drop(0, axis=1)
scores_df = scores_df.rename(columns=col, index=index_model)
scores_df.style.apply(highlight_max)

In [ ]:
param_test ={'num_leaves': sp_randint(6, 50), 
             'min_child_samples': sp_randint(100, 500), 
             'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3, 1e4],
             'subsample': sp_uniform(loc=0.2, scale=0.8), 
             'colsample_bytree': sp_uniform(loc=0.4, scale=0.6),
             'reg_alpha': [0, 1e-1, 1, 2, 5, 7, 10, 50, 100],
             'reg_lambda': [0, 1e-1, 1, 5, 10, 20, 50, 100]}

fit_params={"early_stopping_rounds":30, 
            "eval_metric" : 'auc', 
            "eval_set" : [(X_test,y_test)],
            'eval_names': ['valid'],
            'verbose': 100,
            'categorical_feature': 'auto'}



clf = LGBMClassifier(max_depth=-1, random_state=15, silent=True, metric='None', n_jobs=4, n_estimators=5000)
gs = RandomizedSearchCV(
    estimator=clf, param_distributions=param_test, 
    n_iter=100,
    scoring='roc_auc',
    cv=3,
    refit=True,
    random_state=15,
    verbose=True)


# Uncomment to perform Randomsearch
# gs.fit(X_train, y_train, **fit_params)
# print('Best score reached: {} with params: {} '.format(gs.best_score_, gs.best_params_))

Randomsearch_params = {'colsample_bytree': 0.6261473679815167, 'min_child_samples': 237, 'min_child_weight': 0.001, 'num_leaves': 28, 'reg_alpha': 10, 'reg_lambda': 10, 'subsample': 0.7567691135431514} 

In [ ]:
def learning_rate_010_decay_power_0995(current_iter):
    base_learning_rate = 0.1
    lr = base_learning_rate  * np.power(.995, current_iter)
    return lr if lr > 1e-3 else 1e-3

#set optimal parameters
clf_sw = LGBMClassifier(**clf.get_params())
clf_sw.set_params(**Randomsearch_params)
clf_sw.fit(X_train,y_train, **fit_params, callbacks=[reset_parameter(learning_rate=learning_rate_010_decay_power_0995)])

## ACCURACY : 86%